In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("features/all_features_extended_rgb_glcm.csv")

# --- Combine LBP bins ---
lbp_cols = [f"lbp_bin_{i}" for i in range(9) if f"lbp_bin_{i}" in df.columns]
if lbp_cols:
    df["lbp_mean"] = df[lbp_cols].mean(axis=1)
    df["lbp_std"] = df[lbp_cols].std(axis=1)
    df["lbp_entropy"] = -(df[lbp_cols] * np.log(df[lbp_cols] + 1e-9)).sum(axis=1)
    df = df.drop(columns=lbp_cols)

# --- Combine Gabor features ---
gabor_mean_cols = [c for c in df.columns if "gabor_mean" in c]
gabor_std_cols = [c for c in df.columns if "gabor_std" in c]
if gabor_mean_cols and gabor_std_cols:
    df["gabor_mean_overall"] = df[gabor_mean_cols].mean(axis=1)
    df["gabor_std_overall"] = df[gabor_std_cols].mean(axis=1)
    df = df.drop(columns=gabor_mean_cols + gabor_std_cols)

# --- Combine HSV ---
if all(col in df.columns for col in ["h_mean", "s_mean", "v_mean"]):
    df["hsv_mean_overall"] = df[["h_mean", "s_mean", "v_mean"]].mean(axis=1)
    df["hsv_std_overall"] = df[["h_std", "s_std", "v_std"]].mean(axis=1)

# --- Save cleaned version ---
out_path = "features/all_features_combined.csv"
df.to_csv(out_path, index=False)
print(f"✅ Combined features saved → {out_path}")


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# === Load your cleaned / combined features ===
df = pd.read_csv("features/all_features_combined.csv")  # path as per your setup

# === Keep only numeric columns ===
numeric_df = df.select_dtypes(include=[np.number]).copy()

# === Drop redundant detailed channels ===
drop_cols = [
    "h_mean", "s_mean", "v_mean",
    "h_std", "s_std", "v_std",
    *[f"lbp_bin_{i}" for i in range(9)],
    *[f"gabor_mean_f{f}" for f in [0.1, 0.2, 0.3, 0.4]],
    *[f"gabor_std_f{f}" for f in [0.1, 0.2, 0.3, 0.4]],
    "r_over_g_mean", "r_over_b_mean", "g_over_b_mean",
    "r_over_g_std", "r_over_b_std", "g_over_b_std",
]
numeric_df = numeric_df.drop(columns=[c for c in drop_cols if c in numeric_df.columns], errors="ignore")

print(f"Initial feature count: {numeric_df.shape[1]}")

# === Handle NaN / Inf ===
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan)
numeric_df = numeric_df.fillna(0)

# === Drop zero-variance columns (constant features) ===
numeric_df = numeric_df.loc[:, numeric_df.std() > 1e-8]
print(f"After cleaning → {numeric_df.shape[1]} features")

# === Scale ===
X_scaled = StandardScaler().fit_transform(numeric_df)

# === Compute VIF safely ===
vif_list = []
for i in range(X_scaled.shape[1]):
    try:
        vif_val = variance_inflation_factor(X_scaled, i)
    except Exception as e:
        print(f"⚠️ Skipped {numeric_df.columns[i]} due to {e}")
        vif_val = np.nan
    vif_list.append(vif_val)

vif_data = pd.DataFrame({
    "feature": numeric_df.columns,
    "VIF": vif_list
}).sort_values(by="VIF", ascending=False)

# === Save results ===
vif_data.to_csv("features/vif_results_reduced_clean.csv", index=False)
print("\n✅ VIF analysis complete — saved to 'features/vif_results_reduced_clean.csv'")
print(vif_data.head(20))


In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# --- Load your dataset ---
df = pd.read_csv("features/all_features_extended_rgb_glcm.csv")

# Keep only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# --- Clean NaN / Inf ---
numeric_df = numeric_df.replace([np.inf, -np.inf], np.nan).dropna(axis=0)
print(f"Initial feature count: {numeric_df.shape[1]}")

def compute_vif(df_num):
    """Compute VIF for all columns in a numeric dataframe"""
    X_scaled = StandardScaler().fit_transform(df_num)
    vif = pd.DataFrame()
    vif["feature"] = df_num.columns
    vif["VIF"] = [variance_inflation_factor(X_scaled, i) for i in range(X_scaled.shape[1])]
    return vif.sort_values(by="VIF", ascending=False)

# --- Iteratively remove features with high VIF (>10) ---
current_df = numeric_df.copy()
threshold = 10
iteration = 1

while True:
    vif_df = compute_vif(current_df)
    max_vif = vif_df["VIF"].max()
    if max_vif <= threshold:
        break

    worst_feature = vif_df.iloc[0]["feature"]
    print(f"Iteration {iteration}: Removing '{worst_feature}' (VIF={max_vif:.2f})")
    current_df = current_df.drop(columns=[worst_feature])
    iteration += 1

print(f"\n✅ Reduced to {current_df.shape[1]} features with VIF ≤ {threshold}")

# --- Save reduced features ---
reduced_df = pd.concat([df[["filename", "grid_index"]], current_df], axis=1)
reduced_df.to_csv("features/all_features_vif_pruned.csv", index=False)

# --- Save final VIF table ---
final_vif = compute_vif(current_df)
final_vif.to_csv("features/final_vif_table.csv", index=False)

print("✅ Saved:")
print(" - Clean feature set → features/all_features_vif_pruned.csv")
print(" - Final VIF summary → features/final_vif_table.csv")
print("\nTop 10 remaining features:")
print(final_vif.head(10))


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, roc_auc_score, roc_curve, auc
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# ---------------------------
# --- LOAD FEATURES & LABELS
# ---------------------------
features_df = pd.read_csv("features/all_features_vif_pruned.csv")
labels_df = pd.read_excel("labels_final.xlsx")  # 64-block labels

# ---------------------------
# --- SELECTED FEATURES
# ---------------------------
selected_features = [
    'lbp_bin_5', 'lbp_bin_3', 'gabor_std_f0.2', 'g_over_b_std', 
    'lbp_bin_2', 'sobel_std_x', 'sobel_std_y', 'lbp_bin_7', 
    'glcm_contrast', 'glcm_correlation'
]

# ---------------------------
# --- CLEAN
# ---------------------------
features_df = features_df.replace([np.inf, -np.inf], np.nan)
features_df[selected_features] = features_df[selected_features].fillna(
    features_df[selected_features].mean()
)

# ---------------------------
# --- MERGE FEATURES + LABELS
# ---------------------------
label_long = labels_df.melt(
    id_vars=["filename"], var_name="grid_index", value_name="label"
)
label_long["grid_index"] = label_long["grid_index"].str.extract(r"(\d+)").astype(int)

merged = features_df.merge(label_long, on=["filename", "grid_index"], how="inner")

X = merged[selected_features].values
y = merged["label"].astype(int).values

# ---------------------------
# --- SCALE
# ---------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ---------------------------
# --- OPTION: SMOTE
# ---------------------------
use_smote = True  # Change to False if you don't want SMOTE

if use_smote:
    smote = SMOTE(random_state=42, k_neighbors=1)
    X_scaled, y = smote.fit_resample(X_scaled, y)
    print("✅ SMOTE applied. Class counts:", np.bincount(y))
else:
    print("✅ SMOTE skipped. Class counts:", np.bincount(y))

# ---------------------------
# --- TRAIN/TEST SPLIT
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

# ---------------------------
# --- XGBOOST MODEL
# ---------------------------
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

# ---------------------------
# --- PREDICTIONS
# ---------------------------
y_train_pred = model.predict(X_train)
y_train_proba = model.predict_proba(X_train)[:, 1]

y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:, 1]

# ---------------------------
# --- METRICS
# ---------------------------
def print_metrics(name, y_true, y_pred, y_proba):
    print(f"\n📌 {name} Metrics:")
    print(classification_report(y_true, y_pred, digits=3))
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}")
    print(f"ROC–AUC: {roc_auc_score(y_true, y_proba):.3f}")

print_metrics("TRAIN", y_train, y_train_pred, y_train_proba)
print_metrics("TEST", y_test, y_test_pred, y_test_proba)

# ---------------------------
# --- ROC CURVES
# ---------------------------
plt.figure(figsize=(7, 6))

fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba)

plt.plot(fpr_train, tpr_train, label=f"Train ROC (AUC = {roc_auc_score(y_train, y_train_proba):.3f})")
plt.plot(fpr_test, tpr_test, label=f"Test ROC (AUC = {roc_auc_score(y_test, y_test_proba):.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — XGBoost")
plt.legend()
plt.grid(True)
plt.show()

# ---------------------------
# --- SAVE PREDICTIONS
# ---------------------------
merged["y_pred"] = model.predict(scaler.transform(merged[selected_features]))
merged.to_csv("xgboost_vif_pruned_predictions.csv", index=False)
print("\n✅ Predictions saved → xgboost_vif_pruned_predictions.csv")

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

# --- PARAMETERS ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
IMG_DIR = "images"
OUT_DIR = "xgb_comparisons"
os.makedirs(OUT_DIR, exist_ok=True)

# --- FILE PATHS ---
FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
LABEL_FILE = "labels_final.xlsx"
PRED_FILE = "xgboost_vif_reduced_predictions.csv"
MERGED_OUT = "xgb_smote_predictions_merged.csv"

print("📂 Loading data...")

# 1. --- FEATURES ---
features_df = pd.read_csv(FEATURE_FILE)
features_df = features_df.replace([np.inf, -np.inf], np.nan)
numeric_cols = features_df.select_dtypes(include=[np.number]).columns
features_df[numeric_cols] = features_df[numeric_cols].fillna(features_df[numeric_cols].mean())

# 2. --- LABELS ---
labels_df = pd.read_excel(LABEL_FILE)
label_long = labels_df.melt(
    id_vars=["filename"],
    var_name="block",
    value_name="label"
)
label_long["grid_index"] = label_long["block"].str.extract(r"(\d+)").astype(int)
label_long.drop(columns=["block"], inplace=True)

# 3. --- PREDICTIONS ---
preds_df = pd.read_csv(PRED_FILE)

# Some prediction CSVs already have 'label' and 'y_pred'; if not, fix it
if "y_pred" not in preds_df.columns:
    # If model predictions aren't present, add dummy column
    preds_df["y_pred"] = 0

print("\n📊 Columns before merge:")
print("Predictions:", preds_df.columns.tolist())
print("Labels:", label_long.columns.tolist())

# --- MERGE ---
merged = preds_df.merge(label_long, on=["filename", "grid_index"], how="inner")
print(f"\n✅ Merge completed: {len(merged)} rows matched")

# --- FIX DUPLICATE LABEL COLUMNS ---
if "label_x" in merged.columns and "label_y" in merged.columns:
    print("🧩 Found duplicate label columns → keeping label_y (from Excel labels).")
    merged.rename(columns={"label_y": "label", "label_x": "pred_label_feature"}, inplace=True)
elif "label_x" in merged.columns:
    merged.rename(columns={"label_x": "label"}, inplace=True)
elif "label_y" in merged.columns:
    merged.rename(columns={"label_y": "label"}, inplace=True)

# --- CHECK MERGED DATA ---
print(f"🧾 Columns after merge: {merged.columns.tolist()}")

available_cols = [c for c in ["filename", "grid_index", "label", "y_pred"] if c in merged.columns]
merged[available_cols].to_csv(MERGED_OUT, index=False)
print(f"✅ Saved merged results → {MERGED_OUT}")

if merged.empty or "label" not in merged.columns or "y_pred" not in merged.columns:
    raise ValueError("❌ Merge failed — label or y_pred missing. Check filename/grid_index consistency.")

# --- HELPER FUNCTIONS ---
def enforce_4_3_aspect_and_scale(img):
    """Ensure 4:3 aspect ratio and resize."""
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    return cv2.resize(img, (TARGET_W, TARGET_H))

def draw_grid_overlay(img, grid_vals):
    """Draw 8x8 grid overlay — red=animal(1), green=background(0)."""
    img_vis = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = img_vis.copy()

    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(grid_vals):
                continue
            color = (0, 0, 255) if grid_vals[idx] == 1 else (0, 255, 0)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)
    cv2.addWeighted(overlay, 0.3, img_vis, 0.7, 0, img_vis)
    return img_vis

def draw_comparison(img, labels, preds):
    """Side-by-side: left=ground truth, right=predictions (🟩 correct / 🟥 wrong)."""
    gt_img = draw_grid_overlay(img, labels)
    pred_img = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = pred_img.copy()

    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(preds):
                continue
            color = (0, 255, 0) if preds[idx] == labels[idx] else (0, 0, 255)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)
    cv2.addWeighted(overlay, 0.3, pred_img, 0.7, 0, pred_img)
    combined = np.concatenate((gt_img, pred_img), axis=1)
    return combined

def compute_iou(y_true, y_pred):
    """Intersection over Union."""
    intersection = np.logical_and(y_true == 1, y_pred == 1).sum()
    union = np.logical_or(y_true == 1, y_pred == 1).sum()
    return intersection / union if union > 0 else np.nan

# --- MAIN LOOP ---
print("\n🎨 Generating visual comparisons and computing metrics...")

all_metrics = []

for fname in merged["filename"].unique():
    img_path = os.path.join(IMG_DIR, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing image: {fname}")
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img = enforce_4_3_aspect_and_scale(img)
    subset = merged.loc[merged["filename"] == fname].sort_values("grid_index")
    labels = subset["label"].astype(int).values
    preds = subset["y_pred"].astype(int).values

    if len(labels) < GRID_ROWS * GRID_COLS:
        continue

    combined = draw_comparison(img, labels, preds)

    # --- Compute metrics ---
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    iou = compute_iou(labels, preds)
    all_metrics.append((fname, acc, f1, iou))

    # --- Save and show ---
    out_path = os.path.join(OUT_DIR, fname)
    cv2.imwrite(out_path, combined)

    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title(f"{fname}\nLeft: Ground Truth | Right: Predictions (🟩 correct / 🟥 wrong)\nAcc={acc:.3f}, F1={f1:.3f}, IoU={iou:.3f}")
    plt.axis("off")
    plt.show()

# --- SAVE METRICS ---
metrics_df = pd.DataFrame(all_metrics, columns=["filename", "accuracy", "f1", "iou"])
metrics_df.to_csv("xgb_image_level_metrics.csv", index=False)

print("\n✅ All done!")
print(f"🖼️ Comparisons saved in: {OUT_DIR}/")
print("📊 Per-image metrics saved → xgb_image_level_metrics.csv")
print("\n📈 Mean performance across all images:")
print(metrics_df[["accuracy", "f1", "iou"]].mean().to_frame('mean').T)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# --- FILE PATHS ---
FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
MERGED_FILE = "xgb_smote_predictions_merged.csv"

# --- LOAD ---
features = pd.read_csv(FEATURE_FILE)
merged = pd.read_csv(MERGED_FILE)

for df in [features, merged]:
    df.columns = df.columns.str.strip()

df = features.merge(
    merged[["filename", "grid_index", "label", "y_pred"]],
    on=["filename", "grid_index"],
    how="inner"
)
print(f"🔗 Combined shape after merge: {df.shape}")

# --- SELECT FEATURES ---
exclude_cols = {"filename", "grid_index", "label", "y_pred"}
feature_cols = [c for c in df.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

print(f"🧮 Found {len(feature_cols)} numeric feature columns.")

X = df[feature_cols].values
y_true = df["label"].astype(int).values
y_pred = df["y_pred"].astype(int).values

# --- FIX MISSING VALUES ---
print(f"⚠️ Missing values before imputation: {np.isnan(X).sum()}")
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)
print(f"✅ Missing values after imputation: {np.isnan(X_imputed).sum()}")

# --- SCALE ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# --- PCA ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"📊 PCA explained variance (2D): {pca.explained_variance_ratio_.sum():.3f}")

# --- KMEANS ---
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_pca)
sil_score = silhouette_score(X_pca, clusters)
print(f"🤖 Silhouette score: {sil_score:.3f}")

# --- VISUALIZE ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap="coolwarm", alpha=0.6, s=15)
axes[0].set_title("True Labels (Ground Truth)")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_pred, cmap="coolwarm", alpha=0.6, s=15)
axes[1].set_title("Model Predictions")

axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap="viridis", alpha=0.6, s=15)
axes[2].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', s=100, marker='X', label='Centers')
axes[2].legend()
axes[2].set_title(f"KMeans Clusters (Silhouette={sil_score:.3f})")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.suptitle("PCA + Clustering Visualization (with NaN Fix)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from imblearn.over_sampling import SMOTE

# --- FILE PATHS ---
FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
MERGED_FILE = "xgb_smote_predictions_merged.csv"

# --- LOAD ---
features = pd.read_csv(FEATURE_FILE)
merged = pd.read_csv(MERGED_FILE)

for df in [features, merged]:
    df.columns = df.columns.str.strip()

df = features.merge(
    merged[["filename", "grid_index", "label", "y_pred"]],
    on=["filename", "grid_index"],
    how="inner"
)
print(f"🔗 Combined shape after merge: {df.shape}")

# --- SELECT FEATURES ---
exclude_cols = {"filename", "grid_index", "label", "y_pred"}
feature_cols = [c for c in df.columns if c not in exclude_cols]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]

print(f"🧮 Found {len(feature_cols)} numeric feature columns.")

X = df[feature_cols].values
y_true = df["label"].astype(int).values
y_pred = df["y_pred"].astype(int).values

# --- FIX MISSING VALUES ---
print(f"⚠️ Missing values before imputation: {np.isnan(X).sum()}")
imputer = SimpleImputer(strategy="median")
X_imputed = imputer.fit_transform(X)
print(f"✅ Missing values after imputation: {np.isnan(X_imputed).sum()}")

# --- SCALE ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

# --- APPLY SMOTE ---
print("🧬 Applying SMOTE to balance classes...")
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_scaled, y_true)

print(f"✅ Before SMOTE: {np.bincount(y_true)}")
print(f"✅ After SMOTE:  {np.bincount(y_res)}")

# --- PCA (on SMOTE data) ---
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_res)
print(f"📊 PCA explained variance (2D): {pca.explained_variance_ratio_.sum():.3f}")

# --- KMEANS ---
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_pca)
sil_score = silhouette_score(X_pca, clusters)
print(f"🤖 Silhouette score: {sil_score:.3f}")

# --- VISUALIZE ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# True labels (SMOTE-balanced)
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_res, cmap="coolwarm", alpha=0.6, s=15)
axes[0].set_title("True Labels (SMOTE-balanced)")

# Model predictions (subset for visual consistency)
axes[1].scatter(X_pca[:len(y_pred), 0], X_pca[:len(y_pred), 1],
                c=y_pred, cmap="coolwarm", alpha=0.6, s=15)
axes[1].set_title("Model Predictions")

# Clusters
axes[2].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap="viridis", alpha=0.6, s=15)
axes[2].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='black', s=100, marker='X', label='Centers')
axes[2].legend()
axes[2].set_title(f"KMeans Clusters (Silhouette={sil_score:.3f})")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.suptitle("PCA + KMeans Visualization (with SMOTE)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import pairwise_distances
import matplotlib.pyplot as plt

# --- FILE PATHS ---
FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
LABEL_FILE = "labels_final.xlsx"
IMG_DIR = "images"
OUT_DIR = "smote_visualization"
os.makedirs(OUT_DIR, exist_ok=True)

# --- LOAD DATA ---
features_df = pd.read_csv(FEATURE_FILE)
labels_df = pd.read_excel(LABEL_FILE)

# Melt wide labels into long
labels_long = labels_df.melt(
    id_vars=["filename"],
    var_name="block",
    value_name="label"
)
labels_long["grid_index"] = labels_long["block"].str.extract(r"(\d+)").astype(int)
labels_long.drop(columns=["block"], inplace=True)

# Merge to align features with labels
df = features_df.merge(labels_long, on=["filename", "grid_index"], how="inner")
print(f"✅ Merged data: {df.shape}")

# --- CLEAN FEATURE MATRIX ---
df = df.replace([np.inf, -np.inf], np.nan)
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())
df = df.dropna(subset=num_cols)
print(f"✅ Cleaned data: {df.shape} after dropping NaNs")

# --- PREPARE FEATURES ---
X = df[num_cols].values
y = df["label"].astype(int).values
filenames = df["filename"].values
grid_indices = df["grid_index"].values

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- APPLY SMOTE ---
smote = SMOTE(random_state=42, k_neighbors=2)
X_res, y_res = smote.fit_resample(X_scaled, y)
print(f"🧮 SMOTE created {X_res.shape[0] - X.shape[0]} synthetic samples")

# --- FIND NEAREST REAL NEIGHBORS FOR SYNTHETIC SAMPLES ---
n_new = X_res.shape[0] - X.shape[0]
X_synthetic = X_res[-n_new:]
dists = pairwise_distances(X_synthetic, X_scaled)
closest_idx = np.argmin(dists, axis=1)

# --- GRID PARAMETERS ---
GRID_ROWS, GRID_COLS = 8, 8

def draw_grid(img, grid_vals):
    """Overlay 8x8 grid colored by labels (red=animal, green=background)."""
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = img.copy()

    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(grid_vals):
                continue
            color = (0, 0, 255) if grid_vals[idx] == 1 else (0, 255, 0)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)

    return cv2.addWeighted(overlay, 0.3, img, 0.7, 0)

# --- GENERATE AND SAVE VISUALIZATIONS ---
saved_info = []
for fname in sorted(df["filename"].unique()):
    img_path = os.path.join(IMG_DIR, fname)
    if not os.path.exists(img_path):
        continue

    img = cv2.imread(img_path)
    if img is None:
        continue

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    grid_vals = df.loc[df["filename"] == fname].sort_values("grid_index")["label"].astype(int).values
    if len(grid_vals) < GRID_ROWS * GRID_COLS:
        continue

    img_grid = draw_grid(img_rgb, grid_vals)
    out_path = os.path.join(OUT_DIR, fname)
    cv2.imwrite(out_path, cv2.cvtColor(img_grid, cv2.COLOR_RGB2BGR))

    saved_info.append({
        "filename": fname,
        "animal_cells": int(np.sum(grid_vals == 1)),
        "background_cells": int(np.sum(grid_vals == 0))
    })

print(f"\n✅ Saved {len(saved_info)} visualized images to '{OUT_DIR}'")

# --- SAVE METADATA CSV ---
meta_df = pd.DataFrame(saved_info)
meta_df.to_csv("smote_visualized_images.csv", index=False)
print("📊 Metadata saved → smote_visualized_images.csv")

print("\n🖼️ Example image preview:")
example_file = os.path.join(OUT_DIR, saved_info[0]['filename'])
img_show = cv2.cvtColor(cv2.imread(example_file), cv2.COLOR_BGR2RGB)
plt.imshow(img_show)
plt.title(f"{saved_info[0]['filename']} | 🟥=Animal, 🟩=Background")
plt.axis("off")
plt.show()

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# --- PARAMETERS ---
GRID_ROWS, GRID_COLS = 8, 8
TARGET_W, TARGET_H = 800, 600
IMG_DIR = "images"
PRED_FILE = "xgboost_vif_reduced_predictions.csv"
LABEL_FILE = "labels_final.xlsx"
OUT_DIR = "xgb_visual_eval"
os.makedirs(OUT_DIR, exist_ok=True)

# --- LOAD PREDICTIONS + LABELS ---
print("📂 Loading files...")
preds_df = pd.read_csv(PRED_FILE)
labels_df = pd.read_excel(LABEL_FILE)

# Melt label file (wide ➜ long)
labels_long = labels_df.melt(
    id_vars=["filename"],
    var_name="block",
    value_name="label"
)
labels_long["grid_index"] = labels_long["block"].str.extract(r"(\d+)").astype(int)
labels_long.drop(columns=["block"], inplace=True)

# Merge predictions with true labels
merged = preds_df.merge(labels_long, on=["filename", "grid_index"], how="inner")

if "y_pred" not in merged.columns:
    raise ValueError("❌ Missing 'y_pred' column in predictions!")

print(f"✅ Merged {len(merged)} grid cells across {merged['filename'].nunique()} images")

# --- HELPER FUNCTIONS ---
def enforce_4_3_aspect_and_scale(img):
    """Ensure 4:3 aspect ratio and resize."""
    h, w = img.shape[:2]
    desired_ratio = 4 / 3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    return cv2.resize(img, (TARGET_W, TARGET_H))

def draw_grid_overlay(img, grid_vals):
    """Draw 8×8 grid overlay (red=animal, green=background)."""
    img_vis = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = img_vis.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(grid_vals):
                continue
            color = (0, 0, 255) if grid_vals[idx] == 1 else (0, 255, 0)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)
    cv2.addWeighted(overlay, 0.3, img_vis, 0.7, 0, img_vis)
    return img_vis

def draw_comparison(img, labels, preds):
    """Left: ground truth, Right: prediction (🟩 correct / 🟥 wrong)."""
    gt_img = draw_grid_overlay(img, labels)
    pred_img = img.copy()
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    overlay = pred_img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if idx >= len(preds):
                continue
            color = (0, 255, 0) if preds[idx] == labels[idx] else (0, 0, 255)
            y0, y1 = i * cell_h, (i + 1) * cell_h
            x0, x1 = j * cell_w, (j + 1) * cell_w
            cv2.rectangle(overlay, (x0, y0), (x1, y1), color, -1)
    cv2.addWeighted(overlay, 0.3, pred_img, 0.7, 0, pred_img)
    return np.concatenate((gt_img, pred_img), axis=1)

def compute_iou(y_true, y_pred):
    inter = np.logical_and(y_true == 1, y_pred == 1).sum()
    union = np.logical_or(y_true == 1, y_pred == 1).sum()
    return inter / union if union > 0 else np.nan

# --- MAIN LOOP ---
print("\n🎨 Generating visual comparisons + metrics...")
all_metrics = []

for fname, subset in merged.groupby("filename"):
    img_path = os.path.join(IMG_DIR, fname)
    if not os.path.exists(img_path):
        print(f"⚠️ Missing image: {fname}")
        continue
    img = cv2.imread(img_path)
    if img is None:
        continue

    img = enforce_4_3_aspect_and_scale(img)
    labels = subset.sort_values("grid_index")["label"].astype(int).values
    preds = subset.sort_values("grid_index")["y_pred"].astype(int).values
    if len(labels) < GRID_ROWS * GRID_COLS:
        continue

    combined = draw_comparison(img, labels, preds)

    # --- METRICS ---
    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    iou = compute_iou(labels, preds)
    all_metrics.append([fname, acc, prec, rec, f1, iou])

    # --- SAVE VISUAL ---
    out_path = os.path.join(OUT_DIR, f"{os.path.splitext(fname)[0]}_compare.jpg")
    cv2.imwrite(out_path, combined)

    # --- OPTIONAL: inline plot (comment out if many images) ---
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title(f"{fname}\nAcc={acc:.3f}, Prec={prec:.3f}, Rec={rec:.3f}, F1={f1:.3f}, IoU={iou:.3f}")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# --- SAVE METRICS ---
metrics_df = pd.DataFrame(all_metrics, columns=["filename", "accuracy", "precision", "recall", "f1_score", "iou"])
metrics_df.to_csv("xgb_per_image_metrics.csv", index=False)

print("\n✅ Saved per-image visual results →", OUT_DIR)
print("✅ Saved per-image metrics → xgb_per_image_metrics.csv")
print("\n📈 Mean performance across all images:")
print(metrics_df[["accuracy","precision","recall","f1_score","iou"]].mean().to_frame('mean').T)


In [ ]:
# ============================================================
# XGBoost + SMOTE Semantic Block Classification
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ============================================================
# 1️⃣ Merge Features + Labels
# ============================================================

FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
LABEL_FILE = "labels_final.xlsx"

features = pd.read_csv(FEATURE_FILE)
labels = pd.read_excel(LABEL_FILE)

# Convert wide → long
labels_long = labels.melt(
    id_vars=["filename"],
    var_name="block",
    value_name="label"
)
labels_long["grid_index"] = labels_long["block"].str.extract(r"(\d+)").astype(int)
labels_long.drop(columns=["block"], inplace=True)

# Merge with features
df = features.merge(labels_long, on=["filename", "grid_index"], how="inner")

# Clean data
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(df.mean(numeric_only=True))

print(f"✅ Merged dataset: {df.shape}")
print("📋 Columns:", df.columns[:10].tolist(), "...")

# ============================================================
# 2️⃣ Feature Selection
# ============================================================
selected_features = [
    "sobel_mean_y", "lap_mean", "lap_std", "lap_skew",
    "glcm_contrast", "glcm_homogeneity", "glcm_energy",
    "glcm_correlation", "glcm_dissimilarity",
    "lbp_bin_0", "lbp_bin_1", "lbp_bin_2",
    "h_mean", "s_mean", "v_mean",
    "gabor_mean_f0.1", "gabor_std_f0.1"
]

available_features = [f for f in selected_features if f in df.columns]
print(f"✅ Using {len(available_features)} selected features.")

X = df[available_features].values
y = df["label"].astype(int).values

# ============================================================
# 3️⃣ Preprocessing + SMOTE balancing
# ============================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nClass distribution before SMOTE:", np.bincount(y))
smote = SMOTE(random_state=42, k_neighbors=1)
X_bal, y_bal = smote.fit_resample(X_scaled, y)
print("Class distribution after SMOTE:", np.bincount(y_bal))

# ============================================================
# 4️⃣ Train–Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.25, random_state=42, stratify=y_bal
)

# ============================================================
# 5️⃣ XGBoost Model Training
# ============================================================
model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric="logloss"
)

print("\n🚀 Training XGBoost model...")
model.fit(X_train, y_train)

# ============================================================
# 6️⃣ Evaluation
# ============================================================
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
report = classification_report(y_test, y_pred, digits=3)

print("\n🔹 XGBoost + SMOTE Results:")
print(report)
print(f"Accuracy: {acc:.3f}")
print(f"ROC–AUC: {roc_auc:.3f}")

# ============================================================
# 7️⃣ Confusion Matrix
# ============================================================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix — XGBoost + SMOTE")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ============================================================
# 8️⃣ ROC Curve
# ============================================================
fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — XGBoost + SMOTE")
plt.legend()
plt.grid(True)
plt.show()

# ============================================================
# 9️⃣ Save Predictions
# ============================================================
df["y_pred"] = model.predict(scaler.transform(df[available_features]))
out_path = "xgboost_vif_reduced_predictions.csv"
df.to_csv(out_path, index=False)
print(f"\n💾 Saved predictions → {out_path}")


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# ============================================================
# 1️⃣ LOAD DATA
# ============================================================
FEATURE_FILE = "features/all_features_extended_rgb_glcm.csv"
LABEL_FILE = "labels_final.xlsx"
IMG_DIR = "images"
OUT_DIR = "xgb_highlight_visuals"
os.makedirs(OUT_DIR, exist_ok=True)

features = pd.read_csv(FEATURE_FILE)
labels = pd.read_excel(LABEL_FILE)

# Convert labels wide → long
labels_long = labels.melt(id_vars=["filename"], var_name="block", value_name="label")
labels_long["grid_index"] = labels_long["block"].str.extract(r"(\d+)").astype(int)
labels_long.drop(columns=["block"], inplace=True)

# Merge features with labels
df = features.merge(labels_long, on=["filename", "grid_index"], how="inner")
df = df.replace([np.inf, -np.inf], np.nan).fillna(df.mean(numeric_only=True))

print(f"✅ Merged dataset: {df.shape}")

# ============================================================
# 2️⃣ IMAGE-LEVEL TRAIN / TEST SPLIT (Fix leakage)
# ============================================================
filenames = df["filename"].unique()
train_files, test_files = train_test_split(filenames, test_size=0.25, random_state=42, shuffle=True)

train_df = df[df["filename"].isin(train_files)].copy()
test_df  = df[df["filename"].isin(test_files)].copy()

print(f"Train blocks: {len(train_df)} | Test blocks: {len(test_df)}")

# ============================================================
# 3️⃣ SELECT FEATURES
# ============================================================
selected_features = [
    "sobel_mean_y", "lap_mean", "lap_std", "lap_skew",
    "glcm_contrast", "glcm_homogeneity", "glcm_energy",
    "glcm_correlation", "glcm_dissimilarity",
    "lbp_bin_0", "lbp_bin_1", "lbp_bin_2",
    "h_mean", "s_mean", "v_mean",
    "gabor_mean_f0.1", "gabor_std_f0.1"
]

train_df = train_df.dropna(subset=["label"])
test_df  = test_df.dropna(subset=["label"])

X_train = train_df[selected_features].values
y_train = train_df["label"].astype(int).values

X_test = test_df[selected_features].values
y_test = test_df["label"].astype(int).values

# ============================================================
# 4️⃣ SCALING + SMOTE (train only)
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("\nBefore SMOTE:", np.bincount(y_train))
X_train, y_train = SMOTE(random_state=42, k_neighbors=1).fit_resample(X_train, y_train)
print("After SMOTE:", np.bincount(y_train))

# ============================================================
# 5️⃣ TRAIN XGBOOST
# ============================================================
model = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric="logloss"
)

print("\n🚀 Training XGBoost...")
model.fit(X_train, y_train)

from sklearn.metrics import precision_score, recall_score

# ============================================================
# 🔍 TRAIN METRICS
# ============================================================
y_train_pred = model.predict(X_train)
y_train_proba = model.predict_proba(X_train)[:,1]

acc_train = accuracy_score(y_train, y_train_pred)
prec_train = precision_score(y_train, y_train_pred, zero_division=0)
rec_train = recall_score(y_train, y_train_pred, zero_division=0)
f1_train = f1_score(y_train, y_train_pred)
roc_auc_train = roc_auc_score(y_train, y_train_proba)

print("\n📊 **TRAINING SET METRICS**")
print(f"Accuracy     : {acc_train:.3f}")
print(f"Precision    : {prec_train:.3f}")
print(f"Recall       : {rec_train:.3f}")
print(f"F1 Score     : {f1_train:.3f}")
print(f"ROC–AUC      : {roc_auc_train:.3f}")

# Confusion Matrix (Train)
cm_train = confusion_matrix(y_train, y_train_pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm_train, annot=True, fmt="d", cmap="Greens")
plt.title("✅ Confusion Matrix — TRAIN")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ROC Curve (Train)
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba)
plt.figure(figsize=(6,5))
plt.plot(fpr_train, tpr_train, label=f"AUC = {roc_auc_train:.3f}")
plt.plot([0,1],[0,1],"k--")
plt.legend()
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC Curve — TRAIN")
plt.grid(True)
plt.show()


# ============================================================
# 6️⃣ METRICS
# ============================================================
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
y_test_proba = model.predict_proba(X_test)[:,1]

print("\n✅ Performance:")
print(f"Train  — Acc: {accuracy_score(y_train, y_train_pred):.3f} | F1: {f1_score(y_train, y_train_pred):.3f}")
print(f"Test   — Acc: {accuracy_score(y_test, y_test_pred):.3f} | F1: {f1_score(y_test, y_test_pred):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_test_proba):.3f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (Test Set)")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, y_test_proba):.3f}")
plt.plot([0,1],[0,1],'k--')
plt.legend(); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curve — XGBoost")
plt.show()

# ============================================================
# 7️⃣ VISUALIZATION — FIXED GRID DRAW (row-major)
# ============================================================
GRID_ROWS, GRID_COLS = 8, 8

def draw_highlight(img, labels, preds):
    img = cv2.resize(img, (800,800))
    cell_h, cell_w = img.shape[0]//GRID_ROWS, img.shape[1]//GRID_COLS
    overlay = img.copy()
    
    for idx in range(len(labels)):
        r, c = divmod(idx, GRID_COLS)
        y0, y1 = r*cell_h, (r+1)*cell_h
        x0, x1 = c*cell_w, (c+1)*cell_w

        color = (0,255,0) if preds[idx]==labels[idx] else (0,0,255)
        cv2.rectangle(overlay, (x0,y0),(x1,y1), color, -1)

    return cv2.addWeighted(overlay, 0.35, img, 0.65, 0)

# Generate per-image results (test set only!)
for fname in test_df["filename"].unique():
    sub = test_df[test_df["filename"]==fname].sort_values("grid_index")
    img_path = os.path.join(IMG_DIR, fname)
    if not os.path.exists(img_path): continue

    img = cv2.imread(img_path)
    labels = sub["label"].values
    preds = sub.apply(lambda r: model.predict(scaler.transform([r[selected_features]]))[0], axis=1).values

    vis = draw_highlight(img, labels, preds)
    cv2.imwrite(f"{OUT_DIR}/{fname}", vis)

print("\n✅ Visualization complete — check:", OUT_DIR)


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# ============================================================
# 1) LOAD MERGED FEATURES (with laplacian + GLCM)
# ============================================================
DF = pd.read_csv("merged_features.csv")  # <-- Latest corrected file
DF = DF.replace([np.inf, -np.inf], np.nan).fillna(DF.mean(numeric_only=True))

assert "label" in DF.columns, "❌ 'label' column is missing — check merge step!"

print(f"✅ Data ready: {DF.shape}")

# ============================================================
# 2) FEATURE SELECTION (Santhosh + Laplacian added)
# ============================================================
USE_FEATURES = [
    'row','col','block_id',
    'sal_avg','sal_std',
    'h_mean','s_mean','v_mean',
    'h_std','s_std','v_std',
    'sal_avg_nbrmean','sal_avg_nbrstd',
    'sal_std_nbrmean','sal_std_nbrstd',
    'h_mean_nbrmean','h_mean_nbrstd',
    's_mean_nbrmean','s_mean_nbrstd',
    'v_mean_nbrmean','v_mean_nbrstd',
    'h_std_nbrmean','h_std_nbrstd',
    's_std_nbrmean','s_std_nbrstd',
    'v_std_nbrmean','v_std_nbrstd',
    'sobel_mean_mag','sobel_std_mag',
    'sobel_mean_x','sobel_std_x',
    'sobel_mean_y','sobel_std_y',
    'sobel_mean_mag_nbrmean','sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean','sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean','sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean','sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean','sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean','sobel_std_y_nbrstd',
    'hs_ratio','hv_contrast',
    'lap_mean','lap_std'
]

USE_FEATURES = [f for f in USE_FEATURES if f in DF.columns]
print(f"✅ Using {len(USE_FEATURES)} features.")

# ============================================================
# 3) SPLIT BY IMAGE (NO LEAKAGE) ✅
# ============================================================
image_groups = DF["filename"].unique()
train_imgs, test_imgs = train_test_split(image_groups, test_size=0.25, random_state=42)

train_df = DF[DF["filename"].isin(train_imgs)].copy()
test_df  = DF[DF["filename"].isin(test_imgs)].copy()

print(f"Train blocks: {train_df.shape}, Test blocks: {test_df.shape}")

X_train = train_df[USE_FEATURES].values
y_train = train_df["label"].astype(int).values

X_test = test_df[USE_FEATURES].values
y_test = test_df["label"].astype(int).values

# ============================================================
# 4) SCALE (NO SMOTE) ✅
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\n⚖ Class distribution (TRAIN):", np.bincount(y_train))
print("⚖ Class distribution (TEST) :", np.bincount(y_test))

# ============================================================
# 5) TRAIN XGBOOST WITH CLASS-WEIGHT (NO SMOTE) ✅
# ============================================================
pos_weight = (y_train==0).sum() / (y_train==1).sum()

model = XGBClassifier(
    n_estimators=350,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    eval_metric="logloss"
)

print("\n🚀 Training XGBoost...")
model.fit(X_train, y_train)

# ============================================================
# 6) EVALUATION
# ============================================================
def evaluate(X, y, name):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:,1]

    print(f"\n📊 {name} Performance:")
    print("Accuracy:", accuracy_score(y, y_pred))
    print("F1 Score:", f1_score(y, y_pred, zero_division=0))
    print("ROC-AUC :", roc_auc_score(y, y_proba))

    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(name + " Confusion Matrix")
    plt.show()

evaluate(X_train, y_train, "TRAIN")
evaluate(X_test, y_test, "TEST")

# ============================================================
# 7) SAVE ROC CURVE
# ============================================================
y_proba = model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, y_proba):.3f}")
plt.plot([0,1],[0,1],'k--')
plt.legend()
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — XGBoost (No SMOTE)")
plt.grid()
plt.show()


In [ ]:
import pandas as pd

# --- Load both tables ---
df1 = pd.read_csv("grid_features.csv")  # base grid features
df2 = pd.read_csv("features/all_features_extended_rgb_glcm.csv")  # ← corrected path

# Ensure common keys are aligned
if "grid_index" not in df1.columns and "block_id" in df1.columns:
    df1 = df1.rename(columns={"block_id": "grid_index"})

required_cols = ["filename", "grid_index"]
assert all(col in df1.columns for col in required_cols), "df1 missing filename/grid_index"
assert all(col in df2.columns for col in required_cols), "df2 missing filename/grid_index"

# Select only Laplacian + GLCM features
lap_glcm_cols = [
    "lap_mean", "lap_std",
    "glcm_contrast", "glcm_homogeneity", "glcm_energy",
    "glcm_correlation", "glcm_dissimilarity"
]

available_cols = [c for c in lap_glcm_cols if c in df2.columns]
df2_small = df2[["filename", "grid_index"] + available_cols]

# Merge on filename + grid_index
merged = df1.merge(df2_small, on=["filename", "grid_index"], how="left")

# Fill missing numeric values
merged = merged.fillna(merged.mean(numeric_only=True))

merged.to_csv("merged_features.csv", index=False)
print("✅ merged_features.csv created successfully!")
print("Final shape:", merged.shape)
print("Added feature columns:", available_cols)


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier

# ============================================================
# 1) LOAD MERGED DATA (must contain filename, grid_index, label)
# ============================================================
DF = pd.read_csv("merged_features.csv")
print("✅ Data ready:", DF.shape)

# Ensure no missing values
DF = DF.replace([np.inf, -np.inf], np.nan)
DF = DF.fillna(DF.mean(numeric_only=True))

# ============================================================
# 2) FEATURE SET (as requested)
# ============================================================
USE_FEATURES = [
    'row', 'col', 'grid_index',
    'sal_avg', 'sal_std',
    'h_mean', 's_mean', 'v_mean',
    'h_std', 's_std', 'v_std',
    'sal_avg_nbrmean', 'sal_avg_nbrstd',
    'sal_std_nbrmean', 'sal_std_nbrstd',
    'h_mean_nbrmean', 'h_mean_nbrstd',
    's_mean_nbrmean', 's_mean_nbrstd',
    'v_mean_nbrmean', 'v_mean_nbrstd',
    'h_std_nbrmean', 'h_std_nbrstd',
    's_std_nbrmean', 's_std_nbrstd',
    'v_std_nbrmean', 'v_std_nbrstd',
    'sobel_mean_mag', 'sobel_std_mag',
    'sobel_mean_x', 'sobel_std_x',
    'sobel_mean_y', 'sobel_std_y',
    'sobel_mean_mag_nbrmean', 'sobel_mean_mag_nbrstd',
    'sobel_std_mag_nbrmean',  'sobel_std_mag_nbrstd',
    'sobel_mean_x_nbrmean',   'sobel_mean_x_nbrstd',
    'sobel_std_x_nbrmean',    'sobel_std_x_nbrstd',
    'sobel_mean_y_nbrmean',   'sobel_mean_y_nbrstd',
    'sobel_std_y_nbrmean',    'sobel_std_y_nbrstd',
    'hs_ratio', 'hv_contrast',
    'lap_mean', 'lap_std'
]

USE_FEATURES = [f for f in USE_FEATURES if f in DF.columns]
print(f"✅ Using {len(USE_FEATURES)} features.")

# ============================================================
# 3) SPLIT BY IMAGE (NO LEAKAGE 🔥)
# ============================================================
img_label = DF.groupby('filename')['label'].max().reset_index().rename(columns={'label':'img_has_animal'})
train_imgs, test_imgs = train_test_split(img_label, test_size=0.25, stratify=img_label['img_has_animal'], random_state=42)

train_df = DF[DF['filename'].isin(train_imgs['filename'])]
test_df  = DF[DF['filename'].isin(test_imgs['filename'])]

print(f"Train blocks: {train_df.shape}, Test blocks: {test_df.shape}")

# ============================================================
# 4) SCALE FEATURES (fit only on train → correct ✅)
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[USE_FEATURES])
X_test  = scaler.transform(test_df[USE_FEATURES])

y_train = train_df['label'].astype(int)
y_test  = test_df['label'].astype(int)

# ============================================================
# 5) RANDOMIZED SEARCH CV — XGB TUNING 🎯
# ============================================================
param_dist = {
    "n_estimators": [100, 200, 300, 400],
    "learning_rate": np.linspace(0.01, 0.2, 8),
    "max_depth": [3, 4, 5, 6, 7],
    "subsample": np.linspace(0.7, 1.0, 4),
    "colsample_bytree": np.linspace(0.7, 1.0, 4),
    "gamma": [0, 0.5, 1],
    "min_child_weight": [1, 3, 5]
}

model = XGBClassifier(eval_metric="logloss", random_state=42, tree_method="hist")

search = RandomizedSearchCV(
    model, param_distributions=param_dist,
    scoring="f1", n_iter=30, cv=3, n_jobs=-1, verbose=2
)

print("\n🚀 Running Hyperparameter Search...")
search.fit(X_train, y_train)
best_model = search.best_estimator_

print("\n✅ Best Params:", search.best_params_)

# ============================================================
# 6) EVALUATE BEST MODEL 📊
# ============================================================
def report(split, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob)
    print(f"\n📊 {split} Performance:")
    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {auc:.3f}")

y_train_pred = best_model.predict(X_train)
y_train_prob = best_model.predict_proba(X_train)[:,1]
report("TRAIN", y_train, y_train_pred, y_train_prob)

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:,1]
report("TEST", y_test, y_test_pred, y_test_prob)

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (TEST)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
plt.plot(fpr, tpr, label="ROC Curve")
plt.plot([0,1],[0,1],"k--")
plt.title("ROC Curve (TEST)")
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# Load merged data (correct filename)
df = pd.read_csv("merged_features.csv")

# Features WITH GLCM
USE_FEATURES_GLCM = [
    'sal_avg','sal_std',
    'h_mean','s_mean','v_mean',
    'h_std','s_std','v_std',
    'sobel_mean_mag','sobel_std_mag',
    'sobel_mean_x','sobel_std_x',
    'sobel_mean_y','sobel_std_y',
    'lap_mean','lap_std',
    'glcm_contrast_y','glcm_homogeneity_y','glcm_energy_y',
    'glcm_correlation_y','glcm_dissimilarity_y'
]

df = df.dropna(subset=['label'])
X_full = df[USE_FEATURES_GLCM]
y_full = df['label'].astype(int)
image_ids = df['filename']

# Split by IMAGE to avoid leakage
img_label = df.groupby('filename')['label'].max().reset_index()
img_train, img_test = train_test_split(img_label, test_size=0.25, random_state=42, stratify=img_label['label'])

train_df = df[df['filename'].isin(img_train['filename'])]
test_df  = df[df['filename'].isin(img_test['filename'])]

X_train = train_df[USE_FEATURES_GLCM].values
y_train = train_df['label'].values
X_test  = test_df[USE_FEATURES_GLCM].values
y_test  = test_df['label'].values

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Lower complexity XGBoost to reduce overfitting
model_glcm = XGBClassifier(
    n_estimators=250,
    learning_rate=0.08,
    max_depth=4,
    min_child_weight=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model_glcm.fit(X_train, y_train)

# Eval
y_pred_train = model_glcm.predict(X_train)
y_pred_test = model_glcm.predict(X_test)
y_proba_test = model_glcm.predict_proba(X_test)[:,1]

print("\n📊 WITH GLCM — TRAIN:")
print("Acc:", accuracy_score(y_train,y_pred_train), "F1:", f1_score(y_train,y_pred_train))

print("\n📊 WITH GLCM — TEST:")
print("Acc:", accuracy_score(y_test,y_pred_test), "F1:", f1_score(y_test,y_pred_test),
      "ROC-AUC:", roc_auc_score(y_test,y_proba_test))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba_test)
plt.plot(fpr,tpr,label="GLCM Model")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC — With GLCM")
plt.grid(True); plt.legend(); plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

# -------------------------------------------------------------------
# 1) Load merged data
# -------------------------------------------------------------------
df = pd.read_csv("merged_features.csv")

assert "filename" in df.columns and "grid_index" in df.columns and "label" in df.columns, \
    "❌ dataset must contain filename, grid_index, label columns"

# -------------------------------------------------------------------
# 2) Feature set WITHOUT GLCM
# -------------------------------------------------------------------
USE_FEATURES_NO_GLCM = [
    'sal_avg','sal_std',
    'h_mean','s_mean','v_mean',
    'h_std','s_std','v_std',
    'sobel_mean_mag','sobel_std_mag',
    'sobel_mean_x','sobel_std_x',
    'sobel_mean_y','sobel_std_y',
    'lap_mean','lap_std'
]

print(f"✅ Using {len(USE_FEATURES_NO_GLCM)} features (NO GLCM):")
print(USE_FEATURES_NO_GLCM)

# -------------------------------------------------------------------
# 3) Split by IMAGE (Prevents leakage ✅)
# -------------------------------------------------------------------
# At image level: label = 1 if any block contains animal
image_labels = df.groupby("filename")["label"].max().reset_index()
image_labels.rename(columns={"label": "has_animal"}, inplace=True)

train_imgs, test_imgs = train_test_split(
    image_labels,
    test_size=0.25,
    random_state=42,
    stratify=image_labels["has_animal"]
)

train_df = df[df["filename"].isin(train_imgs["filename"])]
test_df  = df[df["filename"].isin(test_imgs["filename"])]

print(f"\nTrain blocks: {train_df.shape}, Test blocks: {test_df.shape}")
print("Train class distribution:", np.bincount(train_df["label"]))
print("Test class distribution:", np.bincount(test_df["label"]))

# -------------------------------------------------------------------
# 4) Prepare data
# -------------------------------------------------------------------
X_train = train_df[USE_FEATURES_NO_GLCM].values
y_train = train_df["label"].values

X_test = test_df[USE_FEATURES_NO_GLCM].values
y_test = test_df["label"].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -------------------------------------------------------------------
# 5) Train simpler XGBoost model (reduced complexity)
# -------------------------------------------------------------------
model_no_glcm = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.07,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

print("\n🚀 Training model WITHOUT GLCM...")
model_no_glcm.fit(X_train, y_train)

# -------------------------------------------------------------------
# 6) Evaluate
# -------------------------------------------------------------------
train_pred = model_no_glcm.predict(X_train)
test_pred = model_no_glcm.predict(X_test)

train_proba = model_no_glcm.predict_proba(X_train)[:,1]
test_proba = model_no_glcm.predict_proba(X_test)[:,1]

# TRAIN
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred)
train_auc = roc_auc_score(y_train, train_proba)

# TEST
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred)
test_auc = roc_auc_score(y_test, test_proba)

print("\n📊 WITHOUT GLCM — TRAIN:")
print(f"Acc: {train_acc:.3f} | F1: {train_f1:.3f} | ROC-AUC: {train_auc:.3f}")

print("\n📊 WITHOUT GLCM — TEST:")
print(f"Acc: {test_acc:.3f} | F1: {test_f1:.3f} | ROC-AUC: {test_auc:.3f}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load merged dataset
df = pd.read_csv("merged_features.csv")

# Confirm correct columns exist
assert {'filename','grid_index','label'}.issubset(df.columns)

# Feature set WITH GLCM
FEATURES_GLCM = [
    'sal_avg','sal_std',
    'h_mean','s_mean','v_mean',
    'h_std','s_std','v_std',
    'sobel_mean_mag','sobel_std_mag',
    'sobel_mean_x','sobel_std_x',
    'sobel_mean_y','sobel_std_y',
    'lap_mean','lap_std',
    'glcm_contrast_y','glcm_homogeneity_y','glcm_energy_y',
    'glcm_correlation_y','glcm_dissimilarity_y'
]

df = df.dropna()
X_full = df[FEATURES_GLCM].values
y_full = df["label"].astype(int)
image_ids = df["filename"]

# ---------------- Image-level split (no leakage) ----------------
img_label = df.groupby("filename")["label"].max().reset_index()
train_imgs, test_imgs = train_test_split(
    img_label["filename"], test_size=0.25, random_state=42, stratify=img_label["label"]
)

train_df = df[df["filename"].isin(train_imgs)]
test_df  = df[df["filename"].isin(test_imgs)]

X_train = train_df[FEATURES_GLCM].values
y_train = train_df["label"].astype(int).values
X_test  = test_df[FEATURES_GLCM].values
y_test  = test_df["label"].astype(int).values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# ---------------- Hyperparameter Search ----------------
param_grid = {
    'max_depth': [3,4,5,6,7,8],
    'learning_rate': np.linspace(0.01, 0.25, 10),
    'n_estimators': [200,300,400,500],
    'subsample': np.linspace(0.7, 1.0, 5),
    'colsample_bytree': np.linspace(0.7, 1.0, 5),
    'min_child_weight': [1,2,3,4,5],
    'gamma': [0, 0.1, 0.2, 0.3]
}

xgb = XGBClassifier(eval_metric="logloss", random_state=42)

search = RandomizedSearchCV(
    xgb,
    param_distributions=param_grid,
    n_iter=30,
    scoring="f1",
    cv=3,
    verbose=1,
    n_jobs=-1
)

print("\n🚀 Running Randomized Search...")
search.fit(X_train, y_train)

best_model = search.best_estimator_
print("\n✅ Best Parameters:", search.best_params_)

# ---------------- Evaluation ----------------
def evaluate(model, X, y, name):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:,1]
    acc = accuracy_score(y,pred)
    f1 = f1_score(y,pred)
    auc = roc_auc_score(y,proba)
    print(f"\n📊 {name} Performance:")
    print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | ROC-AUC: {auc:.3f}")
    return pred, proba

pred_train, proba_train = evaluate(best_model, X_train, y_train, "TRAIN")
pred_test,  proba_test  = evaluate(best_model, X_test, y_test, "TEST")

# ---------------- Confusion Matrix ----------------
cm = confusion_matrix(y_test, pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix — Test")
plt.show()

# ---------------- ROC Curve ----------------
fpr, tpr, _ = roc_curve(y_test, proba_test)
plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC — Test")
plt.show()

# ---------------- Feature Importance ----------------
plt.figure(figsize=(8,6))
sns.barplot(x=best_model.feature_importances_, y=FEATURES_GLCM)
plt.title("Feature Importance")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os

IMAGE_DIR = "images"  # path to original images
CSV_FILE = "xgboost_vif_reduced_predictions.csv"  # your predictions CSV
OUTPUT_DIR = "predicted_overlays"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GRID_SIZE = 8  # number of blocks along one axis

# Colors for highlighting
BLOCK_COLOR_CORRECT = "green"
BLOCK_COLOR_WRONG = "red"

# Load predictions
import pandas as pd
pred_df = pd.read_csv(CSV_FILE)

for filename in pred_df["filename"].unique():
    img_path = os.path.join(IMAGE_DIR, filename)
    img = Image.open(img_path)
    w, h = img.size
    block_w = w / GRID_SIZE
    block_h = h / GRID_SIZE

    fig, ax = plt.subplots(figsize=(6,6))
    ax.imshow(img)

    # Draw rectangles for each block
    img_blocks = pred_df[pred_df["filename"] == filename]
    for _, row in img_blocks.iterrows():
        i = row['grid_index'] // GRID_SIZE
        j = row['grid_index'] % GRID_SIZE
        color = BLOCK_COLOR_CORRECT if row['label'] == row['y_pred'] else BLOCK_COLOR_WRONG
        rect = patches.Rectangle(
            (j*block_w, i*block_h),
            block_w, block_h,
            linewidth=2,
            edgecolor=color,
            facecolor='none'
        )
        ax.add_patch(rect)

    ax.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"{filename}_overlay.png"), dpi=150)
    plt.close()